<div style="display: flex; align-items: center;">
        <img src="https://synthesizer-project.github.io/synference/_static/synference_logo.png" width="100" style="margin-right: 15px;" alt="Synference Logo">
        <h1 style="margin: 0; font-size: 2em; color: #24292e;">Synference Intro</h1>
        <div style="display: flex; gap: 15px;">
        <a href="https://synthesizer-project.github.io/synference/index.html" target="_blank" style="padding: 6px 12px; background-color: #0366d6; color: white; text-decoration: none; border-radius: 5px; font-weight: 600; font-size: 14px;">Documentation</a>
        <a href="https://github.com/synthesizer-project/synference" target="_blank" style="padding: 6px 12px; background-color: #24292e; color: white; text-decoration: none; border-radius: 5px; font-weight: 600; font-size: 14px;">GitHub</a></div></div>

<body>

</body>





This notebook introduces **synference**, a Python framework for galaxy photometry inference
using SBI, based on **synthesizer** (see Thursday's workshop for more info on
**synthesizer**).

The goal of **synference** is to make it easier to apply SBI techniques for traditional SED
fitting and enable rapid inference of physical parameters from large photometric surveys.

Synference v1.0 is stable and validated in our recent
[paper](https://academic.oup.com/mnras/article/547/1/stag282/8472650?login=false)
but there may still be minor issues and edge cases; we plan to add further features including
more complex noise models, better spectroscopic support, and transformer-based SBI models.
Please report any issues on the
[Github issues page](https://github.com/synthesizer-project/synference/issues).

This notebook covers:
1. **Training our own toy model** — library generation and SBI training from scratch
2. **Using a pre-trained model** — inference on real JADES JWST photometry
3. **Extension: Bring Your Own Data** — using the SPHINX simulation as the forward model

Run the cell below to install Synference (may take a few minutes on Colab).

In [ ]:
import os
os.environ['WITH_OPEMP'] = '1'

try:
  import synference
except ImportError:
  !git clone https://github.com/maho3/ltu-ili.git
  %pip install './ltu-ili[pytorch]'
  %pip install synference
  !git clone https://github.com/synthesizer-project/synthesizer.git
  %pip install './synthesizer'
  !pip install git+https://github.com/WillJRoper/dense_basis.git

import synference

Now we must download the test grid and pretrained model for synthesizer and synference.

In [ ]:
from synthesizer import DATA_DIR
from synference import test_data_dir
import os
%cd {DATA_DIR}

if not os.path.exists('/root/.local/share/Synthesizer/grids/test_grid.hdf5'):
  !synthesizer-download --test-grids --dust-grid
os.makedirs(test_data_dir, exist_ok=True)
if not os.path.exists(f'{test_data_dir}/example_model_library.hdf5'):
  !synference-download --test --destination {test_data_dir}

# Our First Model

Here we will train a simple SBI model that predicts stellar mass, star-formation rate and
dust attenuation from optical/near-IR galaxy photometry. We are limited by Colab's resources,
but feel free to scale things up using the docs!

In synference we deliberately separate **library generation** (making the training data) from
**model training**. One library can power many models with different features, priors or
architectures.

---

## Section 1: Library Generation

Generating a library means running our physical forward model over a grid of input parameters
and storing the resulting synthetic photometry. This library is the **training set** for our
neural density estimator.

Steps:
1. **Sample parameters from a prior** — define the physical ranges to cover
2. **Build stellar population components** — star formation histories and metallicity distributions
3. **Set up the SPS grid and emission model** — physics → light
4. **Define an instrument** — which photometric filters do we observe through?
5. **Run the library** — generate all SEDs and photometry

In [ ]:
from synference import GalaxyBasis

### Step 1: Sampling from the Prior

We need to decide which physical parameters our model covers and draw samples from our prior.
Each sample becomes one synthetic galaxy in the library.

Synference provides `draw_from_hypercube`, which uses **Latin Hypercube Sampling (LHS)** — a
low-discrepancy scheme that covers the parameter space more uniformly than random sampling.
LHS divides each dimension into equal-probability bins and guarantees exactly one sample per
bin combination, so there are no clumps or gaps.

We define a `(min, max)` range for each of six physical parameters:

| Parameter | Range | Physical meaning |
|---|---|---|
| `redshift` | 0 – 5 | Cosmological redshift |
| `log_stellar_mass` | 8 – 12 | log₁₀(M★ / M☉) |
| `log_zmet` | −4 – −1.4 | log₁₀(stellar metallicity) |
| `peak_age_norm` | 0 – 0.99 | SFH peak as fraction of the age of the universe at z |
| `tau` | 0.1 – 2.0 dex | Width of the log-normal SFH |
| `tau_v` | 0 – 3 mag | V-band dust optical depth |

In [ ]:
import matplotlib.pyplot as plt
from synference import draw_from_hypercube

Ngal = 2000

param_dict = {
    "redshift":         (0.0, 5.0),
    "log_stellar_mass": (8.0, 12.0),
    "log_zmet":         (-4.0, -1.4),
    "peak_age_norm":    (0.0, 0.99),  # fraction of the age of the universe at each galaxy's z
    "tau":              (0.1, 2.0),   # dex — width of the log-normal SFH
    "tau_v":            (0.0, 3.0),   # mag — V-band optical depth
}

params = draw_from_hypercube(param_dict, Ngal, rng=42)

# Visualise the prior coverage — LHS should fill the space without clumps or gaps
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(params["redshift"], params["log_stellar_mass"], s=2, alpha=0.5)
axes[0].set_xlabel("Redshift")
axes[0].set_ylabel(r"$\log_{10}(M_\star/M_\odot)$")
axes[0].set_title("Prior coverage: mass vs redshift")

axes[1].scatter(params["tau_v"], params["log_zmet"], s=2, alpha=0.5, color="C1")
axes[1].set_xlabel(r"$\tau_V$ (dust)")
axes[1].set_ylabel(r"$\log_{10}(Z)$")
axes[1].set_title("Prior coverage: dust vs metallicity")

plt.tight_layout()
plt.show()
print(f"Library will contain {Ngal} galaxies sampled over {len(param_dict)} parameters")

The scatter plots confirm that LHS covers the space uniformly. Compare this to how random
sampling of the same N would look — LHS is especially advantageous when N is limited, because
the neural density estimator needs examples from all regions of parameter space.

> **🔭 Extension:** Replace LHS with a physically motivated non-uniform prior. For example,
> draw stellar masses from a Schechter mass function, or couple metallicity to mass via a
> mass–metallicity relation. Just build the numpy arrays yourself instead of using
> `draw_from_hypercube`.

### Step 2: Generating Star Formation Histories

Each galaxy needs a full star formation history (SFH). We use a **log-normal SFH** that peaks
at a lookback time `peak_age` with a width `tau`. This flexible form can represent everything
from bursty dwarfs to smoothly declining massive galaxies.

`generate_sfh_basis` instantiates an array of Synthesizer `SFH` objects from our sampled
arrays. Crucially, `peak_age_norm` is a *fraction* of the age of the universe at each
galaxy's redshift, so no galaxy can have a star-formation peak older than the universe!

In [ ]:
from synthesizer.parametric import SFH
from synference import generate_sfh_basis

sfh_models, _ = generate_sfh_basis(
    sfh_type=SFH.LogNormal,
    sfh_param_names=["peak_age_norm", "tau"],
    sfh_param_arrays=[params["peak_age_norm"], params["tau"]],
    sfh_param_units=[None, None],
    redshifts=params["redshift"],
)

print(f"Generated {len(sfh_models)} SFH objects")

In [ ]:
sfh_models[0].plot_sfh()

### Step 3: Metallicity Distributions

Each galaxy also needs a metallicity distribution. We use `ZDist.DeltaConstant` — a delta
function, meaning each galaxy has a single fixed metallicity. More complex models (e.g. a
spread of metallicities within a galaxy via `ZDist.Normal`) exist in Synthesizer.

> **🔭 Extension:** Implement a mass–metallicity relation by computing `log_zmet` as a
> deterministic function of `log_stellar_mass` rather than sampling it independently from
> the prior.

In [ ]:
from synthesizer.parametric import ZDist

zdists = [ZDist.DeltaConstant(log10metallicity=z) for z in params["log_zmet"]]

print(f"Generated {len(zdists)} metallicity distributions")
print(f"Example: {zdists[0]}")

### Step 4: The Stellar Population Synthesis Grid

Synference uses Synthesizer to compute SEDs by interpolating over a pre-computed
**Stellar Population Synthesis (SPS) grid** — a lookup table mapping stellar age, metallicity,
and ionisation conditions to spectra.

We load a BPASS grid (Binary Population and Spectral Synthesis) with a Chabrier 2003 IMF,
post-processed with Cloudy 23.01 for nebular emission lines and continuum.

We also define a custom wavelength array at constant spectral resolution R = 300 using
`generate_constant_R`, which reduces memory usage while retaining enough spectral information
for broad-band photometry. A constant-R grid spaces wavelengths so that λ/Δλ = R everywhere —
analogous to a spectrograph's resolving power.

In [ ]:
from synthesizer.grid import Grid
from unyt import Angstrom
from synference import generate_constant_R

# Wavelength array from UV to mid-IR at constant R = 300
new_lam = generate_constant_R(R=300, start=1 * Angstrom, stop=50_000 * Angstrom)

grid = Grid("test_grid", new_lam=new_lam)

print(f"Grid loaded: {grid}")
print(f"Wavelength range: {new_lam[0]:.0f} to {new_lam[-1]:.0f}")
print(f"Number of wavelength bins: {len(new_lam)}")

### Step 5: The Emission Model

An **emission model** defines the chain of physical processes that convert stellar populations
into an observed SED. `TotalEmission` combines:

- **Stellar emission** — direct stellar light, read off the SPS grid
- **Nebular emission** — ionised gas lines and continuum (computed by Cloudy inside the grid)
- **Dust attenuation** — a `PowerLaw` screen applied as τ(λ) ∝ λ^slope

Notice we do **not** set `tau_v` here — because it varies per galaxy, we pass it as a
`galaxy_params` argument to `GalaxyBasis` below.

> **🔭 Extension:** Swap `PowerLaw(slope=-0.7)` for `Calzetti` (the empirical starburst
> attenuation law) or `SMC` (steeper, favoured at high redshift). You can also build a
> two-component dust model with separate birth-cloud and diffuse ISM attenuation.

In [ ]:
from synthesizer.emission_models import TotalEmission
from synthesizer.emission_models.attenuation import PowerLaw

emission_model = TotalEmission(
    grid=grid,
    dust_curve=PowerLaw(slope=-0.7),
)

# Plot the emission component tree — the root key tells us what to save later
emission_model.plot_emission_tree()

The tree shows: stellar + nebular → dust attenuation → **emergent** SED. The key `"emergent"`
is what we will ask the library generator to store as our observable.

### Step 6: The Instrument

We define which photometric filters we are observing through. Synference uses Synthesizer
`Instrument` objects containing filter transmission curves.

Here we load **JWST NIRCam wide-band** filters — the workhorse of deep JWST surveys like
JADES, CEERS and COSMOS-Web, spanning roughly 0.6 – 5 μm and covering rest-frame UV and
optical out to z ~ 5.

> **🔭 Extension:** Add HST ACS filters alongside NIRCam for extra UV leverage, or load a
> ground-based instrument to simulate Subaru HSC or Euclid VIS observations.

In [ ]:
from synthesizer.instruments import JWSTNIRCamWide

instrument = JWSTNIRCamWide()
print(f"Filters: {instrument.filters.filter_codes}")

### Putting It All Together — Running the Library

We now assemble all components into a `GalaxyBasis` and run the library generation.

Key points:
- `galaxy_params` carries **per-galaxy** attributes (here, `tau_v` for each galaxy's dust
  optical depth). These are set on individual `Galaxy` objects, not globally on the model
- `params_to_ignore` tells the library not to include `max_age` as an inference target,
  since it is a deterministic function of redshift — the code will warn you about redundant
  parameters
- The `parameter_transforms_to_save` dict lets you save derived quantities such as `max_age`
  as supplementary data for later analysis without re-running the library

Internally, `create_mock_library` uses a Synthesizer `Pipeline` to process galaxies in
batches. With `n_proc=1` this runs on one CPU; increase it if you have multiple cores.

In [ ]:
galaxy_params = {"tau_v": params["tau_v"]}

galaxy_basis = GalaxyBasis(
    grid=grid,
    model_name="workshop_model",
    redshifts=params["redshift"],
    emission_model=emission_model,
    instrument=instrument,
    sfhs=sfh_models,
    metal_dists=zdists,
    log_stellar_masses=params["log_stellar_mass"],
    galaxy_params=galaxy_params,
    params_to_ignore=["max_age"],
)

print("GalaxyBasis created successfully")

In [ ]:
from astropy.cosmology import Planck18 as cosmo
from unyt import Myr

def max_age(x):
    """Age of the universe at a given redshift."""
    return cosmo.age(x["redshift"]).to_value("Myr") * Myr

param_transforms = {"max_age": ("max_age", max_age)}

# This takes a few minutes on Colab — generating 2000 synthetic SEDs
galaxy_basis.create_mock_library(
    "workshop_library",
    emission_model_key="emergent",
    overwrite=True,
    parameter_transforms_to_save=param_transforms,
    n_proc=1,
)

print("Library generation complete!")

### Inspecting the Library

The library is saved as an HDF5 file. Key datasets:

- **`parameters`** — the 6 input parameters for each of the 2000 galaxies
- **`photometry`** — synthetic fluxes in the 8 NIRCam wide-band filters
- **`supplementary_parameters`** — derived quantities (here, `max_age`)
- **`model`** — metadata about the emission model and instrument

Importantly, only `parameters` and `photometry` are strictly required for SBI training.
This means you can bring observables from **any** external simulator — as long as you can
produce these two arrays, synference's SBI machinery will work (more on this in the SPHINX
extension!).

In [ ]:
import h5py
from synference import library_folder

with h5py.File(f"{library_folder}/workshop_library.hdf5") as f:
    for dataset in f:
        print(f"- {dataset}")
        for array in f[dataset]:
            print(f"    - {array}")
            if isinstance(f[dataset][array], h5py.Dataset):
                print(f"        shape: {f[dataset][array].shape}")
            elif isinstance(f[dataset][array], h5py.Group):
                print(f"        keys:  {list(f[dataset][array].keys())}")

---

## Section 2: Training an SBI Model

Now that we have a library of synthetic observables and true parameters, we can train a
**Simulation-Based Inference (SBI)** model. The goal is to learn a neural density estimator
$q_\phi(\theta \mid x)$ that, given photometric fluxes $x$, returns a full posterior
distribution over physical parameters $\theta$ — in a single forward pass, for any galaxy.

Synference uses the [LtU-ILI](https://github.com/maho3/ltu-ili) package, which wraps `sbi`
and `lampe` on top of PyTorch. The default estimator is a **Masked Autoregressive Flow (MAF)**,
a type of normalizing flow well-suited to multi-dimensional posteriors with correlations.

Workflow:
1. Load the library into `SBI_Fitter`
2. Create a **feature array** — decide how to represent the photometry for the network
3. **Train** the density estimator
4. **Validate** with calibration diagnostics

In [ ]:
from synference import SBI_Fitter, library_folder

fitter = SBI_Fitter.init_from_hdf5(
    model_name="workshop_model",
    hdf5_path=f"{library_folder}/workshop_library.hdf5",
)

print("Observations available:", fitter.raw_observation_names)
print("Parameters to infer:   ", fitter.parameter_names)

### The Feature Array

Before training we decide how to represent the photometry for the network. This
**feature array** step has a large impact on model performance:

| Option | Argument | Effect |
|---|---|---|
| AB magnitudes | `flux_units="AB"` | Default — log-flux, well-defined for positive fluxes |
| Log nJy | `flux_units="log10 nJy"` | Alternative log-flux representation |
| Asinh magnitudes | `flux_units="asinh"` | Handles non-detections and negative fluxes |
| Normalise | `normalize_method="JWST/NIRCam.F200W"` | Divide all bands by F200W → pure colours |
| Add redshift | `extra_features=["redshift"]` | Include spectroscopic z as a feature |
| Add noise | `scatter_fluxes=True, depths=3*nJy` | Simulate observational noise at training time |
| Drop a filter | `photometry_to_remove=["JWST/NIRCam.F090W"]` | Remove one band |

For this toy example we use the default (AB magnitudes, no noise). In Section 3 you will see
empirical noise models derived directly from real JADES survey data.

In [ ]:
# Default: AB magnitudes, all 8 NIRCam bands, no noise added
fitter.create_feature_array()

print("Feature array shape: ", fitter.feature_array.shape, "  (n_galaxies x n_features)")
print("Parameter array shape:", fitter.parameter_array.shape, "  (n_galaxies x n_params)")

fitter.plot_histogram_feature_array(bins=20)

In [ ]:
fitter.plot_histogram_parameter_array()

The parameter histograms should look approximately uniform — confirming that LHS covered the
prior evenly. The photometry histograms show the AB magnitude range our model spans.

> **🔭 Extension ideas (feature engineering):**
> - `normalize_method="JWST/NIRCam.F200W"` — normalising to a reference band makes the model
>   sensitive to colour rather than absolute brightness, which often improves mass and SFR recovery
> - `extra_features=["redshift"]` — if you have spectroscopic redshifts, adding them as a
>   feature dramatically tightens constraints on all other parameters
> - `scatter_fluxes=True, depths=3*nJy` — training on realistically noisy photometry makes
>   the model more robust when applied to real data

### Training

`run_single_sbi` trains the neural density estimator. Key arguments:

- `hidden_features` — width of each hidden layer (larger = more expressive, slower)
- `num_transforms` — number of autoregressive steps in the MAF (more = more flexible)
- `stop_after_epochs` — early-stopping patience: training halts when validation loss has
  not improved for this many epochs
- `name_append` — suffix added to the saved model filename

With 2000 galaxies and a small architecture, this should complete in **3–8 minutes on Colab CPU**.

In [ ]:
posterior_model, stats = fitter.run_single_sbi(
    name_append="workshop_v1",
    random_seed=42,
    hidden_features=64,
    num_transforms=4,
    stop_after_epochs=20,
)

### Validating the Model

Synference automatically produces a validation suite after training:

- **Loss curve** — training vs. validation loss; look for smooth decrease with no divergence
- **Corner plot** — posterior for one test galaxy: marginal distributions and parameter correlations
- **TARP / coverage plot** — does the 90% credible interval contain the true value 90% of the
  time? Good calibration = diagonal line
- **True vs. MAP** — do maximum a posteriori estimates recover the true parameters?

We can also re-generate these plots and print numerical metrics on demand:

In [ ]:
fitter.plot_loss(overwrite=True)

In [ ]:
fitter.plot_diagnostics()

In [ ]:
# Numerical metrics: TARP, log DPIT, MSE/RMSE/MAE per parameter, R²
fitter.evaluate_model()

> **🔭 Extension ideas for the SBI model:**
> - **More training data**: regenerate with `Ngal = 10_000` — additional simulations reduce
>   posterior bias and improve coverage
> - **Different architecture**: try `backend="lampe"` in `run_single_sbi` to use
>   Flow Matching Posterior Estimation instead of MAF
> - **Hyperparameter search**: synference integrates with [Optuna](https://optuna.org/) for
>   automated tuning — see the
>   [model optimization docs](https://synthesizer-project.github.io/synference/sbi_train/model_optimization.html)
> - **Online / sequential training**: iteratively focus new simulations near the posterior to
>   improve accuracy with fewer total simulations — see the
>   [online training docs](https://synthesizer-project.github.io/synference/sbi_train/online_training.html)

---

# Here's one I trained earlier...

Training with 2000 galaxies gave us a working prototype. For real science we need a
production-quality model trained on a much larger library with realistic noise.

The synference [paper](https://academic.oup.com/mnras/article/547/1/stag282/8472650?login=false)
trained a model using:
- **~500,000** synthetic galaxies from a BPASS SPS grid
- A **Dense Basis** SFH parametrization (more flexible than log-normal)
- **Empirical noise models** derived directly from the JADES photometric catalogue — so the
  noise added during training exactly matches the noise distribution of the real data
- Combined HST ACS + JWST NIRCam photometry

This model was downloaded earlier via `synference-download`. We will now apply it to a subset
of the **JADES** (JWST Advanced Deep Extragalactic Survey) spectroscopic catalogue — one of
the deepest JWST surveys, covering the GOODS-S field with NIRSpec spectroscopic redshifts.

In [ ]:
from synference import SBI_Fitter, load_unc_model_from_hdf5, test_data_dir

# Path to the model library HDF5 (stores training photometry + parameters)
library_path = (
    f"{test_data_dir}/grid_BPASS_Chab_DenseBasis_SFH_0.01_z_14_logN_2.7_Calzetti_v3_multinode.hdf5"
)

# Load the pre-trained SBI model (model_file is the directory containing the saved .pkl)
fitter_pt = SBI_Fitter.load_saved_model(
    model_file=f"{test_data_dir}",
    library_path=library_path,
    device="cpu",
)

# Load empirical noise models — these were fit to the flux vs. flux-error relation
# in the real JADES catalogue for each photometric band
nm_path = f"{test_data_dir}/BPASS_DenseBasis_v4_final_nsf_0_params_empirical_noise_models.h5"
noise_models = load_unc_model_from_hdf5(nm_path)
fitter_pt.feature_array_flags["empirical_noise_models"] = noise_models

print("Pre-trained model loaded!")
print("Parameters the model infers:", fitter_pt.parameter_names)
print("Number of features:         ", len(fitter_pt.feature_names))

### Recreating the Simulator

Synference stores enough metadata in the library HDF5 to reconstruct the original Synthesizer
simulator. This is needed for **SED recovery** later — generating predicted SEDs from posterior
parameter samples to verify that the inferred parameters actually reproduce the observed
photometry.

In [ ]:
fitter_pt.recreate_simulator_from_library(
    override_library_path=library_path,
    override_grid_path="test_grid.hdf5",  # must be in the Synthesizer grids directory
)
print("Simulator recreated successfully")

### Loading the Real JADES Data

We load a subset of the JADES spectroscopic catalogue. Each source has NIRSpec spectroscopic
redshifts and NIRCam + HST ACS photometry.

Having spectroscopic redshifts is a major advantage: we include redshift as a **known feature**
(rather than inferring it), which dramatically tightens constraints on all other parameters —
especially stellar mass and SFR, which are strongly degenerate with distance.

In [ ]:
from astropy.table import Table

cat = Table.read(f"{test_data_dir}/jades_spec_catalogue_subset.fits")

print(f"Catalogue: {len(cat)} galaxies")
print(f"Columns:   {cat.colnames[:20]}...")
cat[:5]

### Mapping Catalogue Columns to Model Features

The pre-trained model expects photometry labeled with SVO filter names
(e.g. `JWST/NIRCam.F200W`), but our catalogue uses short names (e.g. `F200W`).
We build a **conversion dictionary** that maps catalogue column names to the model's expected
feature names.

Flux uncertainty columns are prefixed with `unc_`, and we also map the spectroscopic redshift
column so the model knows which column holds the redshift feature.

In [ ]:
def band_to_instrument(band):
    """Map a short filter name to its SVO-format instrument/filter string."""
    if band in ["F435W", "F606W", "F775W", "F814W", "F850LP"]:
        return f"HST/ACS_WFC.{band}"
    return f"JWST/NIRCam.{band}"

# Extract the band names the model expects (strip unc_ prefix and instrument prefix)
bands = [
    i.split(".")[-1]
    for i in fitter_pt.feature_names
    if not (i.startswith("unc_") or i == "redshift")
]
print("Bands used by model:", bands)

# Map catalogue column → model feature name
conversion_dict = {band: band_to_instrument(band) for band in bands}
conversion_dict.update({f"unc_{band}": f"unc_{band_to_instrument(band)}" for band in bands})
conversion_dict["redshift"] = "redshift"

print("\nConversion dict (first 4 entries):")
for k, v in list(conversion_dict.items())[:4]:
    print(f"  {k!r:25} -> {v!r}")

### Running Inference on the Full Catalogue

`fit_catalogue` applies the trained model to real data. For each galaxy it:
1. Transforms catalogue photometry into model features (applying the noise model normalisation)
2. Draws `num_samples` posterior samples from the trained NDE
3. Computes the 16th, 50th and 84th percentiles of each parameter's marginal posterior
4. Returns an astropy `Table` with columns named `{param}_16`, `{param}_50`, `{param}_84`

We draw 300 posterior samples per galaxy — enough for reliable percentiles. On Colab CPU
this takes roughly 10–30 seconds for the full catalogue.

In [ ]:
from unyt import Jy

post_tab = fitter_pt.fit_catalogue(
    cat,
    columns_to_feature_names=conversion_dict,
    flux_units=Jy,              # catalogue fluxes are in Jy
    check_out_of_distribution=False,
    recover_SEDs=False,
    missing_data_flag=float("nan"),
    num_samples=300,
    append_to_input=False,
)

print(f"Inference complete for {len(post_tab)} galaxies")
print("Output columns:", post_tab.colnames[:12], "...")
post_tab[:5]

Any `nan` rows had missing photometry that we chose not to impute. Set `missing_data_mcmc=True`
in `fit_catalogue` to marginalise over missing bands rather than discarding those galaxies.

### The Star-Forming Main Sequence

One of the most fundamental galaxy scaling relations is the **star-forming main sequence** —
the tight correlation between stellar mass and SFR for star-forming galaxies. Let's plot it
from our SBI posteriors.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mask = np.isfinite(post_tab["log_mass_50"]) & np.isfinite(post_tab["log_sfr_50"])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    post_tab["log_mass_50"][mask],
    post_tab["log_sfr_50"][mask],
    s=15, alpha=0.7,
)
ax.set_xlabel(r"$\log_{10}(M_\star\,/\,M_\odot)$", fontsize=13)
ax.set_ylabel(r"$\log_{10}(\mathrm{SFR}\,/\,M_\odot\,\mathrm{yr}^{-1})$", fontsize=13)
ax.set_title("Star-Forming Main Sequence — JADES (SBI)", fontsize=13)
plt.tight_layout()
plt.show()

### SED Recovery

A unique feature of synference is **SED recovery**: for each galaxy, draw posterior parameter
samples, run each through the Synthesizer simulator, and overlay the resulting SED ensemble
on the observed photometry.

If the model works correctly, the observed photometry should lie within the posterior SED
envelope. Large discrepancies indicate either a poor fit or an out-of-distribution galaxy.

We recover SEDs for the first 3 sources in the catalogue:

In [ ]:
from IPython.display import display

post_tab_sed, sed_data = fitter_pt.fit_catalogue(
    cat[:3],
    columns_to_feature_names=conversion_dict,
    flux_units=Jy,
    check_out_of_distribution=False,
    recover_SEDs=True,
    plot_SEDs=True,
    missing_data_flag=float("nan"),
    num_samples=300,
    append_to_input=False,
)

for key in sed_data:
    display(sed_data[key]["fig"])

The shaded region shows the 16th–84th percentile posterior SED envelope. Points show observed
photometry with error bars. Good agreement means the model has found physically consistent
parameter combinations that reproduce the data.

> **🔭 Extension ideas:**
> - **Outlier detection**: set `check_out_of_distribution=True` to flag galaxies whose
>   photometry lies outside the training distribution (uses [PyOD](https://pyod.readthedocs.io/))
> - **Impute missing bands**: `missing_data_mcmc=True` marginalises over missing photometry
>   rather than discarding those galaxies
> - **Compare to traditional codes**: cross-match your SBI masses with BAGPIPES or CIGALE
>   values from the literature — differences highlight model uncertainty and systematic offsets
> - **Other scaling relations**: plot `dust_50` vs `log_mass_50`, or the mass–metallicity
>   relation, using other posterior median columns

---

## Extension: Bring Your Own Data — Training on SPHINX

The library generation pipeline above assumed Synthesizer as the forward model. But what if
you want to use the outputs of a hydrodynamical simulation — where the "simulator" is a full
cosmological code running on a supercomputer?

Synference provides `LibraryCreator`, which converts **any** external grid of synthetic
observables into the required HDF5 format. This lets you use the full SBI training pipeline
with any simulator.

Here we use the [**SPHINX** public data release](https://github.com/HarleyKatz/SPHINX-20-data)
— mock JWST photometry for ~1,380 galaxies from the SPHINX-20 radiation-hydrodynamic
simulation at z = 4.6–10, observed in 10 different orientations. This demonstrates that
synference is a general-purpose SBI framework, not tied to Synthesizer.

In [ ]:
import os
import numpy as np
from astropy.table import Table
from synference import LibraryCreator, SBI_Fitter

sphinx_url = (
    "https://raw.githubusercontent.com/HarleyKatz/SPHINX-20-data/"
    "refs/heads/main/data/all_basic_data.csv"
)

if not os.path.exists("all_basic_data.csv"):
    os.system(f"wget {sphinx_url} -O all_basic_data.csv")

sphinx = Table.read("all_basic_data.csv")
print(f"SPHINX table: {len(sphinx)} galaxies, {len(sphinx.colnames)} columns")
print("Sample columns:", sphinx.colnames[:10])

In [ ]:
# Use viewing direction 0 (of 10 available orientations)
VIEW = 0

# Physical parameters — these become the SBI inference targets
parameter_columns = [
    "redshift", "stellar_mass", "stellar_metallicity",
    "mean_stellar_age_mass", "sfr_3", "sfr_10", "sfr_100",
    f"ebmv_dir_{VIEW}",
]
parameter_units = [
    "dimensionless", "log10(Msun)", "dimensionless",
    "Myr", "Msun/yr", "Msun/yr", "Msun/yr", "dimensionless",
]

# JWST NIRCam photometry columns present in the SPHINX release
sphinx_filters = [
    "F070W", "F090W", "F115W", "F140M", "F150W", "F162M", "F182M", "F200W",
    "F210M", "F250M", "F277W", "F300M", "F335M", "F356W", "F360M",
    "F410M", "F430M", "F444W", "F460M", "F480M",
]
feature_columns = [f"{f}_dir_{VIEW}" for f in sphinx_filters]

# Supplementary quantities — stored but not inferred by default
supp_columns = [f"fesc_dir_{VIEW}", f"beta_dir_{VIEW}_sn", f"MAB_1500_dir_{VIEW}"]
supp_units   = ["dimensionless", "dimensionless", "AB"]

In [ ]:
# Build numpy arrays with shape (n_params, n_galaxies) as required by LibraryCreator
parameters    = sphinx[parameter_columns].to_pandas().to_numpy().T
features_raw  = sphinx[feature_columns].to_pandas().to_numpy().T
supplementary = sphinx[supp_columns].to_pandas().to_numpy().T

# SPHINX provides AB magnitudes — convert to nJy for synference conventions.
# The AB zeropoint in nJy is 31.4:  m_AB = -2.5 * log10(f / nJy) + 31.4
def mag_to_njy(mag):
    flux = 10 ** (-0.4 * (mag - 31.4))
    flux[mag == 0] = 0              # zeros indicate missing / non-detections
    flux[~np.isfinite(flux)] = 0
    return flux

features_njy = mag_to_njy(features_raw)

# Use SVO-format filter names so synference can match them to instrument objects
svo_names = [f"JWST/NIRCam.{f}" for f in sphinx_filters]

print(f"Parameter array shape: {parameters.shape}   (n_params × n_galaxies)")
print(f"Feature array shape:   {features_njy.shape}  (n_filters × n_galaxies)")

In [ ]:
# Build the synference HDF5 library from the SPHINX arrays
LibraryCreator(
    model_name="SPHINX_JWST",
    parameter_grid=parameters,
    observation_grid=features_njy,
    observation_names=svo_names,
    observation_units="nJy",
    parameter_names=parameter_columns,
    parameter_units=parameter_units,
    supplementary_parameters=supplementary,
    supplementary_parameter_names=supp_columns,
    supplementary_parameter_units=supp_units,
    out_folder=".",
    overwrite=True,
)
print("SPHINX library saved to ./library_SPHINX_JWST.h5")

In [ ]:
# Load it back to verify the round-trip
fitter_sphinx = SBI_Fitter.init_from_hdf5(
    model_name="SPHINX_JWST",
    hdf5_path="./library_SPHINX_JWST.h5",
)

print("Parameters:", fitter_sphinx.parameter_names)
print("Filters:   ", fitter_sphinx.raw_observation_names[:5], "...")
print(f"Library size: {fitter_sphinx.parameter_array.shape[0]} galaxies")


From here, call `fitter_sphinx.create_feature_array` and" `fitter_sphinx.run_single_sbi` to train — exactly as in Section 2!

In [ ]:
# your code here!

> **🔭 SPHINX extension ideas:**
> - **Use all 10 orientations**: stack all viewing directions to get ~13,800 training
>   samples and build a model that is robust to galaxy orientation
> - **Orientation as a nuisance parameter**: include `VIEW` as an inference parameter so
>   the model marginalises over orientation when recovering physical properties
> - **Model comparison**: train equivalent SBI models on SPHINX and on the Synthesizer-based
>   library from Section 1. Where do inferred masses and SFRs agree? Discrepancies quantify
>   model uncertainty — important for robust inference
> - **High-z applications**: SPHINX spans z = 4.6–10, making this library valuable for
>   studying reionisation-era galaxies where parametric SFH models may be inadequate

---

Well done, you've finished all the notebook content! If you'd like to explore more, why don't you check out the [docs](https://synthesizer-project.github.io/synference/index.html), which have many more examples of advanced features, such as hyperparameter optimisation using Optuna, custom training loops, or transformer based SBI architectures.